In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Define paths
dataset_dir = "/content/drive/MyDrive/Medicinal plant dataset"  # Update with your actual path
image_size = (224, 224)  # Adjust if needed
batch_size = 32 # Adjust as needed based on your system's memory

# Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # 20% for validation
)

train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training' # Use training subset for training
)


validation_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation' # Use validation subset for validation
)


# Load pre-trained MobileNetV2 model (excluding the top classification layer) 
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(image_size[0], image_size[1], 3))

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x) # Adjust units as needed
predictions = Dense(len(train_generator.class_indices), activation='softmax')(x) # Number of classes

# Create the final model
model = Model(inputs=base_model.input, outputs=predictions)

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(
    train_generator,
    epochs=10, # Adjust number of epochs as needed
    validation_data=validation_generator
)


# Print model summary
model.summary()

# Save the model (optional)
model.save('medicinal_plant_classifier.h5')


# Example of prediction (after training)
# import numpy as np
# from tensorflow.keras.preprocessing import image

# img_path = 'path/to/your/test/image.jpg'
# img = image.load_img(img_path, target_size=image_size)
# x = image.img_to_array(img)
# x = np.expand_dims(x, axis=0)
# x = x / 255.0  # Normalize the image

# preds = model.predict(x)
# class_idx = np.argmax(preds[0])
# class_labels = list(train_generator.class_indices.keys())
# print(f"Predicted class: {class_labels[class_idx]}")


Found 4765 images belonging to 41 classes.
Found 1180 images belonging to 41 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 4198s 28s/step - accuracy: 0.4300 - loss: 2.2295 - val_accuracy: 0.7703 - val_loss: 0.7914
Epoch 2/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 87s 584ms/step - accuracy: 0.8240 - loss: 0.5847 - val_accuracy: 0.8356 - val_loss: 0.5748
Epoch 3/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 85s 569ms/step - accuracy: 0.8875 - loss: 0.3777 - val_accuracy: 0.8754 - val_loss: 0.4425
Epoch 4/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 86s 579ms/step - accuracy: 0.9140 - loss: 0.2739 - val_accuracy: 0.8305 - val_loss: 0.4955
Epoch 5/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 86s 578ms/step - accuracy: 0.9250 - loss: 0.2254 - val_accuracy: 0.8788 - val_loss: 0.4179
Epoch 6/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 86s 578ms/step - accuracy: 0.9417 - loss: 0.1823 - val_accuracy: 0.8576 - val_loss: 0.4792
Epoch 7/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 89s 601ms/step - accuracy: 0.9525 - loss: 0.1568 - val_accuracy: 0.8975 - val_loss: 0.3544
Epoch 8/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 85s 570ms/step - accuracy: 0.9517 - loss: 0

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 6,319,293 (24.11 MB)

 Trainable params: 1,353,769 (5.16 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 2,707,540 (10.33 MB)

In [ ]:


import numpy as np
from tensorflow.keras.preprocessing import image

img_path = '/content/drive/MyDrive/Medicinal plant dataset/Neem/1783.jpg'
img = image.load_img(img_path, target_size=(224, 224))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = x / 255.0  # Normalize the image

preds = model.predict(x)
class_idx = np.argmax(preds[0])
class_labels = list(train_generator.class_indices.keys())
print(f"Predicted class: {class_labels[class_idx]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Predicted class: Neem


In [ ]:


from google.colab import files
files.download('medicinal_plant_classifier.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 7.3 MB/s eta 0:00:00


In [ ]:
class_labels = list(train_generator.class_indices.keys()) # Assuming train_generator is defined in your original code
print(class_labels
      )

['Aloevera', 'Amla', 'Amruta_Balli', 'Arali', 'Ashoka', 'Ashwagandha', 'Avacado', 'Bamboo', 'Basale', 'Betel', 'Betel_Nut', 'Brahmi', 'Castor', 'Curry_Leaf', 'Doddapatre', 'Ekka', 'Ganike', 'Gauva', 'Geranium', 'Henna', 'Hibiscus', 'Honge', 'Insulin', 'Jasmine', 'Lemon', 'Lemon_grass', 'Mango', 'Mint', 'Nagadali', 'Neem', 'Nithyapushpa', 'Nooni', 'Pappaya', 'Pepper', 'Pomegranate', 'Raktachandini', 'Rose', 'Sapota', 'Tulasi', 'Wood_sorel', 'labels']


In [2]:
# prompt: now make a gradio app for upload the img and show the img with the  there predicted class

import gradio as gr
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the trained model
model = tf.keras.models.load_model('/content/drive/MyDrive/medicinal_plant_classifier.h5')

# Load class labels (assuming you have them saved)
# Replace with your actual class labels
class_labels = list(train_generator.class_indices.keys()) # Assuming train_generator is defined in your original code

def predict_image(img):
    img = img.resize((224, 224))  # Resize image to match model input
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Normalize

    preds = model.predict(img_array)
    predicted_class_index = np.argmax(preds[0])
    predicted_class = class_labels[predicted_class_index]
    confidence = preds[0][predicted_class_index]
    return predicted_class, confidence, img


iface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs=[gr.Label(num_top_classes=3), gr.Number(label="Confidence"), gr.Image(type="pil")],
    title="Medicinal Plant Classifier",
    description="Upload an image of a medicinal plant to classify it.",
)

iface.launch(debug=True)

NameError: name 'train_generator' is not defined